In [1]:
import torch
import torch.nn as nn

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding('gpt2')

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.device(device)

device(type='cuda')

In [4]:
gpt_config_for_124M_param = {
    'vocab_size': 50257,
    'context_length': 1024,
    'emb_dim': 768,
    'n_heads': 12,
    'n_layers': 12,
    'drop_rate': 0.1,
    'qkv_bias': False,
}

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, emd_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emd_dim))
        self.shift = nn.Parameter(torch.zeros(emd_dim))
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)

        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        
        return norm_x * self.scale + self.shift



class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))



class Feedforward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4*cfg['emb_dim']),
            nn.GELU(),
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim'])
        )
    
    def forward(self, x):
        return self.layers(x)


In [6]:
class MultiheadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_len, dropout, n_heads, qkv_bias=False):
        super().__init__()
        
        assert (d_out % n_heads == 0), 'd_out must be divisible by n_heads'

        self.d_out = d_out
        self.d_in = d_in
        self.n_heads = n_heads
        self.head_dim = d_out // n_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_len, context_len), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)

        keys = keys.view(b, num_tokens, self.n_heads, self.head_dim)
        query = query.view(b, num_tokens, self.n_heads, self.head_dim)
        value = value.view(b, num_tokens, self.n_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        query = query.transpose(1, 2)
        value = value.transpose(1, 2)

        attn_scores = query @ keys.transpose(2, 3)

        mask = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ value).transpose(1, 2)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        context_vec = self.out_proj(context_vec)

        return context_vec


In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.attn = MultiheadAttention(
            d_in=cfg['emb_dim'],
            d_out=cfg['emb_dim'],
            context_len=cfg['context_length'],
            n_heads=cfg['n_heads'],
            dropout=cfg['drop_rate'],
            qkv_bias=cfg['qkv_bias'],
        )

        self.ff = Feedforward(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.drop_shortcut = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attn(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

In [8]:
x = torch.randn(2, 4, 768)
block = TransformerBlock(gpt_config_for_124M_param)
out = block(x)

out.shape, x.shape

(torch.Size([2, 4, 768]), torch.Size([2, 4, 768]))

In [9]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg['n_layers'])]
        )

        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)
    
    
    def forward(self, in_idx, targets=None):
        device = self.tok_emb.weight.device
        in_idx = in_idx.to(device)
        if targets is not None:
            targets = targets.to(device)
        batch, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.drop_emb(x)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        loss = None

        if targets is not None:
            loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

In [10]:
def text_to_tokens(text):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    
    return encoded_tensor

def tokens_to_text(tokens):
    tokens = tokens.squeeze(0)
    
    return tokenizer.decode(tokens.tolist()) 

In [11]:
class GPTDataLoader:
    def __init__(self, B, T, file_path="input.txt"):
        self.B = B
        self.T = T 

        enc = tiktoken.get_encoding("gpt2")
        
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # Tokenize the text and convert to a PyTorch tensor
        tokens = enc.encode(text)
        self.tokens = torch.tensor(tokens, dtype=torch.long)

        self.current_position = 0
    
    def next_batch(self):
        B, T = self.B, self.T

        # Boundary check: Ensure we have enough tokens left for a full batch (B * T + 1)
        # If we reach the end of the corpus, reset the pointer to the beginning
        if self.current_position + B * T + 1 > self.tokens.size(0):
            self.current_position = 0
        
        # Slice out the buffer of size (B * T + 1)
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        
        # x is the input sequence; y is x shifted by 1 token as target
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)

        # Move the pointer forward by the total number of tokens processed (B * T)
        self.current_position += B * T
        return x, y


In [12]:
def generate_simple_text(model, idx, max_new_tokens,context_size):

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)
            logits = logits[:, -1, :]

            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.argmax(probs, dim=-1, keepdim=True)

            idx = torch.cat((idx, idx_next), dim=1)
        
    return idx

In [15]:
def estimate_loss(model, data_loader, eval_iters=10):
    model.eval() # model in evaluation mode

    losses = []

    for _ in range(eval_iters):
        x, y = data_loader.next_batch()
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            _, loss = model(x, y)

        losses.append(loss.item())
    
    model.train() # model back training mode

    return sum(losses) / len(losses)

# Initializing optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


# Training loop
max_steps = 100

for step in range(max_steps):
    x, y = data_loader.next_batch()
    x, y = x.to(device), y.to(device)

    logits, loss = model(x, y)

    # Backward Pass
    optimizer.zero_grad() # Clear gradients from previous step
    loss.backward()       # Compute gradients of the loss w.r.t model parameters
    optimizer.step()      # Update model parameters

    if step % 10 == 0:
        val_loss = estimate_loss(model, data_loader)
        print(f"Step {step:3d} | Train Loss: {loss.item():.4f} | Est. Loss: {val_loss:.4f}")

: 

In [14]:
model = GPTModel(gpt_config_for_124M_param)
data_loader = GPTDataLoader(4, 256)